# OsteoClassAI v1 (2024)

Notebook original del proyecto de Electromedicina (Universidad de Ibagué), presentado en la 48.ª METIE.

Se conserva tal como se entregó. Único cambio: se restauró la sangría del `for` en `matriz_confusion`, que se perdió al guardar desde Colab.

**Advertencia:** los resultados de esta versión están sobreestimados por fuga de datos entre entrenamiento y prueba. Ver `../README.md` (Lecciones aprendidas) y la v2.


In [ ]:
import os
import numpy as np

from sklearn.model_selection import train_test_split

import tensorflow as tf

from tensorflow.keras import layers, models, optimizers
from keras.layers import Conv2D, Layer, Multiply, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
def load_images_paths_and_labels(data_dir):
    image_paths = []
    labels = []

    for label in [0, 1, 2, 3, 4]:
        for split in ['train', 'test', 'val']:
            folder_path = os.path.join(data_dir, split, str(label))
            for img_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, img_name)

                if os.path.isfile(img_path):
                    image_paths.append(img_path)
                    labels.append(label)

    return np.array(image_paths), np.array(labels)

In [ ]:
sobel_data_dir = 'data_edges_sobel'

classes_int = [0, 1, 2]
classes_str = ['0', '1-2', '3-4']

sobel_x, sobel_y = load_images_paths_and_labels(sobel_data_dir)

X_train_sobel_o, X_temp_sobel_o, y_train_sobel_o, y_temp_sobel_o = train_test_split(sobel_x, sobel_y, test_size = 0.15, random_state = 42)
X_test_sobel_o, X_val_sobel_o, y_test_sobel_o, y_val_sobel_o = train_test_split(X_temp_sobel_o, y_temp_sobel_o, test_size = 0.333, random_state = 42)

print("Tamaño del conjunto de entrenamiento:", X_train_sobel_o.shape)
print("Tamaño del conjunto de prueba:", X_test_sobel_o.shape)
print("Tamaño del conjunto de validación:", X_val_sobel_o.shape)


y_train_sobel = [0 if y == 0 else 1 if y in [1, 2] else 2 for y in y_train_sobel_o]

y_test_sobel = [0 if y == 0 else 1 if y in [1, 2] else 2 for y in y_test_sobel_o]

y_val_sobel = [0 if y == 0 else 1 if y in [1, 2] else 2 for y in y_val_sobel_o]

In [ ]:
def transform(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels = 1)
    image = tf.image.grayscale_to_rgb(image)

    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32)/255.0  # Escalar a [0, 1]
    image = (image - [0.485, 0.456, 0.406])/[0.229, 0.224, 0.225]  # Normalizar según ImageNet

    return image, label

def augment_image(image, label):
    #image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    image = tf.image.random_brightness(image, max_delta = 0.1)
    image = tf.image.random_contrast(image, lower = 0.8, upper = 1.2)

    zoom = tf.random.uniform([], 0.8, 1.2)
    new_size = tf.cast(tf.cast(tf.shape(image)[0:2], tf.float32) * zoom, tf.int32)
    image = tf.image.resize(image, new_size)
    image = tf.image.resize_with_crop_or_pad(image, 224, 224)

    return image, label

In [ ]:
def list_to_dataset(X_train, y_train, X_test, y_test):
    batch_size = 32
    # Convierte las listas de rutas y etiquetas en un Dataset de pares (ruta, etiqueta)

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(lambda path, label: transform(path, label), num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.map(lambda image, label: augment_image(image, label), num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
    test_dataset = test_dataset.map(lambda path, label: transform(path, label), num_parallel_calls = tf.data.AUTOTUNE)
    test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return train_dataset, test_dataset

In [ ]:
class MultiScaleAttention(Layer):
    def __init__(self, kernel_sizes=[3, 5, 7], **kwargs):
        super(MultiScaleAttention, self).__init__(**kwargs)
        self.kernels = [Conv2D(1, kernel_size=k, padding='same', activation='sigmoid') for k in kernel_sizes]

    def call(self, inputs):
        # Aplicar convoluciones de diferentes tamaños
        attention_maps = [conv(inputs) for conv in self.kernels]

        # Fusionar las atenciones a través de la dimensión de canales
        attention_map = tf.reduce_max(tf.stack(attention_maps, axis=-1), axis=-1)

        # Aplicar el mapa de atención sobre la entrada
        return Multiply()([inputs, attention_map])

    def get_config(self):
        # Guardar los parámetros necesarios para la capa
        config = super(MultiScaleAttention, self).get_config()
        config.update({
            "kernel_sizes": self.kernel_sizes,
        })
        return config

class SelfAttention(Layer):
    def __init__(self, num_heads=1, key_dim=64, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.num_heads = num_heads   # Inicialización del número de cabezas de atención
        self.key_dim = key_dim       # Dimensión de las claves/proyecciones
        self.query = Dense(key_dim)  # Proyección para la consulta (query)
        self.key = Dense(key_dim)    # Proyección para la clave (key)
        self.value = Dense(key_dim)  # Proyección para el valor (value)

    def call(self, inputs):
        # Calcular consultas, claves y valores
        q = self.query(inputs)  # (batch_size, height, width, key_dim)
        k = self.key(inputs)    # (batch_size, height, width, key_dim)
        v = self.value(inputs)  # (batch_size, height, width, key_dim)

        # Calculamos la atención utilizando el producto punto
        attention_weights = tf.nn.softmax(tf.matmul(q, k, transpose_b=True))  # (batch_size, height, width, height*width)

        # Aplicamos la atención a los valores
        output = tf.matmul(attention_weights, v)  # (batch_size, height, width, key_dim)
        return output

    def get_config(self):
        # Llama a get_config() de la superclase y añade los parámetros necesarios
        config = super(SelfAttention, self).get_config()
        config.update({
            "num_heads": self.num_heads,
            "key_dim": self.key_dim
        })
        return config

In [ ]:
def train_model(num_classes, train_dataset, test_dataset):

    # Paso 1: Modelo preentrenado
    base_model = tf.keras.applications.VGG19(weights = 'imagenet', include_top = False, input_shape = (224, 224, 3))

    base_model.trainable = False

    model = models.Sequential([
        base_model, # Capas originales
        SelfAttention(),
        MultiScaleAttention(kernel_sizes=[3, 5, 7]), # Aplicar atención espacial
        layers.GlobalAveragePooling2D(), # Capa que reduce las dimensiones manteniendo la info relevante
        layers.BatchNormalization(), # Normalización de lotes
        Dense(512, activation='relu'),
        layers.Dropout(0.33), # Dropout para evitar sobreajuste
        layers.Dense(num_classes, activation = 'linear') # Clasificación multicategoría
    ])

    model.compile(optimizer = optimizers.Adam(learning_rate = 0.0001), #Optimizador y tasa de aprendizaje lr = 0.00005
                  loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), #Clasificación de clases enteras
                  metrics=['accuracy'])

    # Callbacks para reducir la tasa de aprendizaje si el modelo se estanca y hacer early stopping
    lr_reduction = ReduceLROnPlateau(monitor ='loss', factor = 0.2, patience = 3, verbose = 1, min_lr = 1e-6)
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    num_epochs = 50

    history_1 = model.fit(
        train_dataset,
        validation_data = test_dataset,
        epochs = num_epochs,
        callbacks=[lr_reduction, early_stopping],
        verbose=1
    )


    # Paso 2: Fine-tuning - descongelar las últimas capas del modelo base
    base_model.trainable = True  # Descongelar el modelo base
    for layer in base_model.layers[:-10]:  # Congelar todas excepto las últimas 10 capas
        layer.trainable = False

    model.compile(optimizer = optimizers.Adam(learning_rate = 0.0001), #Optimizador y tasa de aprendizaje lr = 0.00005
                  loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), #Clasificación de clases enteras
                  metrics=['accuracy'])

    history_2 = model.fit(
        train_dataset,
        validation_data = test_dataset,
        epochs = num_epochs,
        callbacks=[lr_reduction, early_stopping],
        verbose=1
    )

    return model, history_1, history_2

In [ ]:
def matriz_confusion(test_dataset, model, classes_s, classes_i):
    all_preds = []
    all_labels = []

    for images, labels in test_dataset:

        # Realiza predicciones
        preds = tf.argmax(model(images, training=False), axis=1) # training=False asegura que se use el modo de evaluación

        # Añade las predicciones y etiquetas a las listas
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

    # Convertir listas a arrays de NumPy
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)

    # Calcular la precisión
    accuracy = accuracy_score(all_labels, all_preds)
    print(f'Accuracy on test set: {accuracy:.4f}')

    # Reporte de clasificación
    print('Classification Report:')
    print(classification_report(all_labels, all_preds,  target_names=classes_s, labels=classes_i))

    # Matriz de confusión
    cm = confusion_matrix(all_labels, all_preds)
    print('Confusion Matrix:')
    print(cm)

    # Graficar matriz de confusión
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes_i, yticklabels=classes_i)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
def graficar_entrenamiento(history):
    # Crear una figura con dos subgráficas en una sola fila
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Gráfica de pérdida de entrenamiento y validación
    ax1.plot(history.history['loss'], label='Pérdida de Entrenamiento')
    ax1.plot(history.history['val_loss'], label='Pérdida de Validación')
    ax1.set_title('Pérdida durante el Entrenamiento')
    ax1.set_xlabel('Épocas')
    ax1.set_ylabel('Pérdida')
    ax1.legend()

    # Gráfica de precisión de entrenamiento y validación
    ax2.plot(history.history['accuracy'], label='Precisión de Entrenamiento')
    ax2.plot(history.history['val_accuracy'], label='Precisión de Validación')
    ax2.set_title('Precisión durante el Entrenamiento')
    ax2.set_xlabel('Épocas')
    ax2.set_ylabel('Precisión')
    ax2.legend()

    # Ajustar la disposición de las gráficas
    plt.tight_layout()
    plt.show()

In [ ]:
train_dataset_sobel, test_dataset_sobel = list_to_dataset(X_train_sobel_o, y_train_sobel, X_test_sobel_o, y_test_sobel)

In [ ]:
model_wide, history_wide_1, history_wide_2 = train_model(3, train_dataset_sobel, test_dataset_sobel)
# 80m 55.2s

In [ ]:
with tf.device('/CPU:0'):
    matriz_confusion(test_dataset_sobel, model_wide, classes_str, classes_int)
    # 1m 50.6s

In [ ]:
graficar_entrenamiento(history_wide_1)
graficar_entrenamiento(history_wide_2)

model_wide.save('model_wide_vgg19.h5')

In [ ]:
_, val_dataset_sobel = list_to_dataset(X_train_sobel_o, y_train_sobel, X_val_sobel_o, y_val_sobel)
matriz_confusion(val_dataset_sobel, model_wide, classes_str, classes_int)